## Loading Model and Input Files 

In [1]:
# 🔄 CLEAN INITIALIZATION - RESTART KERNEL IF NEEDED
print("🔄 Initializing environment...")

import google.generativeai as genai
import pathlib
import httpx
import os
import asyncio
import json
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Check if API key is loaded
api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("GOOGLE_API_KEY not found in environment variables. Please check your .env file.")

# Configure the API
genai.configure(api_key=api_key)

# Test if the import is working correctly
try:
    test_model = genai.GenerativeModel('gemini-2.0-flash-exp')
    print("✅ Google Generative AI imported and configured successfully")
except Exception as e:
    print(f"❌ Error with Google Generative AI: {e}")

pa_file = "PA.pdf"
referral_file = "referral_package.pdf"

# Retrieve and encode the PDF byte
filepath = pathlib.Path('Input Data/Adbulla/' + pa_file)

print("✅ Setup complete!")


🔄 Initializing environment...
✅ Google Generative AI imported and configured successfully
✅ Setup complete!


/Users/haroon/Projects /Headstarter/Automate Insurance claims/mandolin-project/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Mistral OCR Data Extraction from refferal Package

In [2]:
from mistralai import Mistral

client = Mistral(api_key=os.getenv("MISTRAL_API_KEY"))

with open('Input Data/Adbulla/referral_package.pdf', "rb") as f:
    file_content = f.read()

uploaded = client.files.upload(
    file={"file_name": referral_file, "content": file_content},
    purpose="ocr"
)
signed = client.files.get_signed_url(file_id=uploaded.id)

ocr = client.ocr.process(
    model="mistral-ocr-latest",
    document={"type": "document_url", "document_url": signed.url},
    include_image_base64=False
)

# Save structured content in a variable
referral_package_text = ""
for pg in ocr.pages:
    referral_package_text += pg.markdown + "\n\n"

# Print structured content
for pg in ocr.pages:
    print(pg.markdown)


04/22/2024 WED 11:32 FAX

001/028

# FAX TRANSMISSION

**Better Life Multiple Sclerosis Center**

3320 Montgomery Dr. Nashville, TN 37361

**BetterLife**

F 615-562-4820 P: 615-562-4848

Dr. Asriel Han | Dr. Aetya Shan

---

**TO:**

Golden Gate Infusion Center

**Fax:** 614-225-3355 **Phone:** 614-295-7655

**From:** Erran Rostami, BSN, RN

**P:** 615-343-1176

**F:** 615-343-1219

---

**Page**

(including cover sheet)

---

**Comments:**

- Arabic - spoken / English - written
- Rituxen (Truxima) TP
- MRI Reports
- Hospital DIC Make
- Demographics

---

The documents accompanying this transmission may contain health information that is legally protected. This information is intended only for the use of the individual or entity named above. The authorized recipient of this information is prohibited from disclosing this information to any other party unless permitted by law or regulation.

If you are not the intended recipient, you are hereby notified that any use, disclosure, copying 

## 📄 Referral Package Analysis & Information Extraction to Json Using Gemeni 


In [3]:
# Enhanced prompt for referral package analysis
referral_prompt = """You are analyzing a referral package (collection of scanned medical documents) to extract patient information that will be used to fill a Prior Authorization form.

CONTEXT: This referral package contains scanned documents like insurance cards, medical history notes, test results, and other supporting documentation. The extracted information will be mapped to specific fields in a PA form.

EXTRACTION FOCUS: Extract all available information that could be relevant for PA form completion, including:

PATIENT DEMOGRAPHICS:
- Full name, DOB, gender, address, phone numbers
- Patient ID numbers, MRN, account numbers
- Emergency contacts and relationships

INSURANCE INFORMATION:
- Insurance company name and plan details
- Member ID, group number, policy number
- Subscriber information and relationship to patient
- Coverage details and effective dates

CLINICAL INFORMATION:
- Primary and secondary diagnoses with ICD codes
- Current medications, dosages, and frequencies
- Allergies and adverse reactions
- Vital signs and lab results
- Previous treatments and outcomes
- Medical history and comorbidities

PROVIDER INFORMATION:
- Referring physician name, specialty, and contact info
- Practice name and address
- NPI numbers and license information
- Facility details where treatment will occur

TREATMENT DETAILS:
- Requested medication/procedure/device
- Dosage, frequency, duration
- Medical necessity justification
- Previous treatment failures
- Clinical criteria met for approval

SUPPORTING EVIDENCE:
- Lab results supporting diagnosis
- Imaging studies and results
- Functional assessments
- Specialist consultations
- Treatment response documentation

Return structured JSON with all extracted information, using null for missing data."""

# Extract structured data from referral package
model = genai.GenerativeModel('gemini-2.0-flash')
referral_response = model.generate_content([
    referral_prompt,
    f"REFERRAL PACKAGE OCR TEXT:\n{referral_package_text}"
])

referral_data = referral_response.text
print("Referral Package Data:")
print(referral_data)

Referral Package Data:
```json
{
  "patient_demographics": {
    "full_name": "Shakh Abdulla",
    "dob": "04/01/2001",
    "gender": "male",
    "address": "425 Sherman Ave, APT D, Nashville TN 37995 and 8327 BROADWAY LN APT D KNOXVILLE, TN 37923",
    "phone_numbers": {
      "home": "865-395-3958",
      "mobile": "865-395-0481"
    },
    "patient_id_numbers": {
      "mrn": "041152153 and 048152153",
      "care_everywhere_id": "VDJ-TKR2-484T-LGF8"
    },
    "emergency_contacts": [
      {
        "name": "Sina, Amin",
        "relationship": "Mother"
      },
      {
        "name": "Mohammedraza, Musiala",
        "relationship": null
      }
    ]
  },
  "insurance_information": {
    "insurance_company_name": "BC TENNCARE",
    "plan_details": "TC BLUE CARE NO COPAY",
    "member_id": "LAJM14345116",
    "group_number": "435000",
    "policy_number": null,
    "subscriber_information": {
      "subscriber_name": "ABDULLA, SHAKH",
      "subscriber_dob": null,
      "relations

## PA widget input info extraction

In [4]:
import fitz

# Code from Headstarter tutorial to extract fields from PDF
def extracting_fields_from_pdf(pdf_file):
    """
    Extracts form fields from a PDF file and returns them as a list of dictionaries.
    Each dictionary contains field name, type, value, page number, and other metadata.
    """
    if not pathlib.Path(pdf_file).exists():
        raise FileNotFoundError(f"The file {pdf_file} does not exist.")
    doc = fitz.open(pdf_file)
    fields = []
    for page_num, page in enumerate(doc, start = 1):
        for w in page.widgets() or []:
            field = {
                "name": w.field_name,
                "type": "checkbox" if w.field_type == fitz.PDF_WIDGET_TYPE_CHECKBOX else "text",
                "value": w.field_value,
                "page": page_num,
                "field_type": w.field_type,
                "field_type_string": w.field_type_string,
                "field_label": w.field_label,
            }
            print(field)
            fields.append(field)

    field_by_page = {}
    for field in fields:
        page_num = field["page"]
        if page_num not in field_by_page:
            field_by_page[page_num] = []
    
    return field_by_page

pa_fields = extracting_fields_from_pdf("Input Data/Adbulla/PA.pdf")


{'name': 'CB1', 'type': 'checkbox', 'value': '', 'page': 2, 'field_type': 2, 'field_type_string': 'CheckBox', 'field_label': 'Start of treatment'}
{'name': 'T2', 'type': 'text', 'value': '', 'page': 2, 'field_type': 7, 'field_type_string': 'Text', 'field_label': 'Start date: (MM)'}
{'name': 'T3', 'type': 'text', 'value': '', 'page': 2, 'field_type': 7, 'field_type_string': 'Text', 'field_label': 'Start date: (DD)'}
{'name': 'T4', 'type': 'text', 'value': '', 'page': 2, 'field_type': 7, 'field_type_string': 'Text', 'field_label': 'Start date: (YYYY)'}
{'name': 'CB5', 'type': 'checkbox', 'value': '', 'page': 2, 'field_type': 2, 'field_type_string': 'CheckBox', 'field_label': 'Continuation of therapy'}
{'name': 'T6', 'type': 'text', 'value': '', 'page': 2, 'field_type': 7, 'field_type_string': 'Text', 'field_label': 'Date of last treatment: (MM)'}
{'name': 'T7', 'type': 'text', 'value': '', 'page': 2, 'field_type': 7, 'field_type_string': 'Text', 'field_label': 'Date of last treatment: (D

## PA feild organization Prompt template for Gemeni 

In [5]:
def format_fields_for_pa(pa_fields):
  return """
  You are given an array of dictionaries, each representing a form field extracted from a PDF. Each dictionary contains:
  - "name": the field name
  - "type": the field type (e.g., "text", "checkbox")
  - "value": the field value (may be null)
  - "page": the 1-based page number
  - "field_type": the PDF widget type code
  - "field_type_string": the PDF widget type as a string
  - "field_label": the field label (may be null)

  TASK:
  1. Organize these fields into a hierarchical JSON tree grouped by page number.
  2. For each page, create an array of fields, preserving all metadata.
  3. The top-level JSON should be:
  {
    "pages": {
      "1": [ ...fields... ],
      "2": [ ...fields... ],
      ...
    }
  }
  4. Do not omit any fields or metadata.
  5. Return only valid JSON (no markdown, no extra commentary).

  Example output:
  {
    "pages": {
      "1": [
        {
          "name": "T1",
          "type": "text",
          "value": "",
          "page": 1,
          "field_type": 2,
          "field_type_string": "Text",
          "field_label": "First Name"
        },
        ...
      ],
      "2": [
        ...
      ]
    }
  }

  <PA_FIELDS>
  {pa_fields}
  </PA_FIELDS> """

## Send Prompt to Gemeni 

In [6]:
# Send to Gemini 

async def query_gemini_async(prompt, pdf_path, model_name='gemini-2.5-flash'):
    """
    Async function to query Gemini with a PDF and prompt using the correct API
    """
    filepath = pathlib.Path(pdf_path)
    
    # Use the correct GenerativeModel approach (same as we tested in cell 1)
    model = genai.GenerativeModel(model_name)
    
    # Create the PDF part for multimodal input
    pdf_part = {
        "mime_type": "application/pdf",
        "data": filepath.read_bytes()
    }
    
    # Run the synchronous API call in an executor to make it async
    loop = asyncio.get_event_loop()
    response = await loop.run_in_executor(
        None,
        lambda: model.generate_content([pdf_part, prompt])
    )
    
    print(response.text)
    return response.text

In [7]:
pa_fields_context = {}

# Process each page of the PA form
async def process_page(page):
    prompt = format_fields_for_pa(pa_fields[page])
    print (prompt)

    result = await query_gemini_async(prompt, "Input Data/Adbulla/PA.pdf")
    print ("---------------------------------")
    return page, result
# create a list of tasks to run in parallel
tasks = [process_page(page) for page in pa_fields]

# running the tasks in parallel
results = await asyncio.gather(*tasks)

# 
for page, result in results:
    pa_fields_context[page] = result



  You are given an array of dictionaries, each representing a form field extracted from a PDF. Each dictionary contains:
  - "name": the field name
  - "type": the field type (e.g., "text", "checkbox")
  - "value": the field value (may be null)
  - "page": the 1-based page number
  - "field_type": the PDF widget type code
  - "field_type_string": the PDF widget type as a string
  - "field_label": the field label (may be null)

  TASK:
  1. Organize these fields into a hierarchical JSON tree grouped by page number.
  2. For each page, create an array of fields, preserving all metadata.
  3. The top-level JSON should be:
  {
    "pages": {
      "1": [ ...fields... ],
      "2": [ ...fields... ],
      ...
    }
  }
  4. Do not omit any fields or metadata.
  5. Return only valid JSON (no markdown, no extra commentary).

  Example output:
  {
    "pages": {
      "1": [
        {
          "name": "T1",
          "type": "text",
          "value": "",
          "page": 1,
          "fiel

In [8]:
# Create PA_form_analysis from pa_fields_context
# This combines all the processed PA field data into a single analysis

if 'pa_fields_context' in locals() and pa_fields_context:
    # Combine all page results into a single analysis
    PA_form_analysis = ""
    for page, result in pa_fields_context.items():
        PA_form_analysis += f"PAGE {page}:\n{result}\n\n"
    print("✅ PA_form_analysis created successfully!")
    print(f"Length: {len(PA_form_analysis)} characters")
else:
    # Fallback: create from raw pa_fields if pa_fields_context isn't ready
    PA_form_analysis = json.dumps(pa_fields, indent=2)
    print("⚠️  Using fallback: PA_form_analysis created from raw pa_fields")
    print("Note: Run cell 12 first to get processed analysis")

print("PA_form_analysis preview:")
print(PA_form_analysis)


✅ PA_form_analysis created successfully!
Length: 275553 characters
PA_form_analysis preview:
PAGE 2:
```json
{
  "pages": {
    "1": [
      {
        "name": "Phone",
        "type": "text",
        "value": "1-866-503-0857",
        "page": 1,
        "field_type": 2,
        "field_type_string": "Text",
        "field_label": null
      },
      {
        "name": "Fax",
        "type": "text",
        "value": "1-844-268-7263",
        "page": 1,
        "field_type": 2,
        "field_type_string": "Text",
        "field_label": null
      },
      {
        "name": "Availity",
        "type": "text",
        "value": "https://www.aetna.com/health-care-professionals/resource-center/availity.html",
        "page": 1,
        "field_type": 2,
        "field_type_string": "Text",
        "field_label": null
      },
      {
        "name": "Phone_2",
        "type": "text",
        "value": "1-855-463-0933",
        "page": 1,
        "field_type": 2,
        "field_type_string": "Tex

In [9]:
pa_fields_context

{2: '```json\n{\n  "pages": {\n    "1": [\n      {\n        "name": "Phone",\n        "type": "text",\n        "value": "1-866-503-0857",\n        "page": 1,\n        "field_type": 2,\n        "field_type_string": "Text",\n        "field_label": null\n      },\n      {\n        "name": "Fax",\n        "type": "text",\n        "value": "1-844-268-7263",\n        "page": 1,\n        "field_type": 2,\n        "field_type_string": "Text",\n        "field_label": null\n      },\n      {\n        "name": "Availity",\n        "type": "text",\n        "value": "https://www.aetna.com/health-care-professionals/resource-center/availity.html",\n        "page": 1,\n        "field_type": 2,\n        "field_type_string": "Text",\n        "field_label": null\n      },\n      {\n        "name": "Phone_2",\n        "type": "text",\n        "value": "1-855-463-0933",\n        "page": 1,\n        "field_type": 2,\n        "field_type_string": "Text",\n        "field_label": null\n      },\n      {\n      

## form filling strategy 

In [10]:
# 🎯 FORM FILLING STRATEGY - Map Patient Data to PA Form Fields
print("🔄 Creating form filling strategy...")

filling_prompt = """You are creating a form-filling strategy for a Prior Authorization form based on available patient data from a referral package.

TASK: Analyze the PA form structure and referral package data to create a mapping strategy that determines:
1. Which patient data maps to which form fields
2. How to format the data for each field type
3. What values to use for checkboxes and selections
4. Which fields can be filled vs. which are missing data

INPUT DATA:
- PA_form_analysis: Contains the structure and fields of the PA form
- referral_data: Contains extracted patient information from medical records

MAPPING REQUIREMENTS:
- Map patient demographics to form fields (name, DOB, address, etc.)
- Map insurance information to appropriate fields
- Map clinical data (diagnosis, medications, provider info)
- Handle date formatting (MM/DD/YYYY format)
- Set appropriate checkbox values based on clinical context
- Identify missing information that needs manual input

OUTPUT FORMAT: Return a JSON mapping strategy with specific field assignments:
{
    "field_mappings": {
        "field_name": {
            "value": "data_to_fill",
            "source": "referral_data_field",
            "confidence": "high|medium|low",
            "field_type": "text|checkbox"
        }
    },
    "missing_data": {
        "required_fields": [],
        "optional_fields": []
    },
    "completion_status": {
        "total_fields": 0,
        "fillable_fields": 0,
        "completion_percentage": 0
    }
}"""

# Create form filling strategy
model = genai.GenerativeModel('gemini-2.0-flash')

strategy_response = model.generate_content([
    filling_prompt,
    f"PA FORM ANALYSIS:\n{PA_form_analysis}",
    f"REFERRAL PACKAGE DATA:\n{referral_data}"
])

filling_strategy = strategy_response.text
print("✅ Form Filling Strategy Created!")
print("Preview:")
print( filling_strategy)


🔄 Creating form filling strategy...
✅ Form Filling Strategy Created!
Preview:
```json
{
    "field_mappings": {
        "Pat_First": {
            "value": "Shakh",
            "source": "referral_data.patient_demographics.full_name",
            "confidence": "high",
            "field_type": "text"
        },
        "Pat_Last": {
            "value": "Abdulla",
            "source": "referral_data.patient_demographics.full_name",
            "confidence": "high",
            "field_type": "text"
        },
        "Pat_DOB": {
            "value": "04/01/2001",
            "source": "referral_data.patient_demographics.dob",
            "confidence": "high",
            "field_type": "text"
        },
        "Pat_Address": {
            "value": "425 Sherman Ave, APT D, Nashville TN 37995 and 8327 BROADWAY LN APT D KNOXVILLE, TN 37923",
            "source": "referral_data.patient_demographics.address",
            "confidence": "high",
            "field_type": "text"
        },
  

In [11]:
# 🤖 INTELLIGENT AUTO-MAPPING: Let AI map actual field names to patient data
print("🤖 Creating intelligent automated field mapping...")

# Get all actual PDF fields with their labels
doc = fitz.open("Input Data/Adbulla/PA.pdf")
pdf_fields = []
for page_num in range(len(doc)):
    page = doc[page_num]
    for widget in page.widgets():
        if widget.field_name and widget.field_label:
            pdf_fields.append({
                'field_name': widget.field_name,
                'field_type': widget.field_type_string,
                'field_label': widget.field_label,
                'page': page_num + 1
            })
doc.close()

print(f"📋 Found {len(pdf_fields)} fields with labels in PDF")

# Create intelligent mapping prompt
auto_mapping_prompt = f"""You are an expert at mapping medical form fields to patient data.

TASK: Create a JSON mapping that connects actual PDF field names to patient data values.

PDF FIELDS (field_name -> field_label):
{json.dumps(pdf_fields[:50], indent=2)}  

PATIENT DATA:
{referral_data[:2000]}

INSTRUCTIONS:
1. Match PDF field names (like "T12", "CB1") to appropriate patient data values
2. Use the field_label to understand what each field expects
3. For text fields, provide the exact string value
4. For checkboxes, provide true/false
5. For date fields, format as needed (MM/DD/YYYY or MM, DD, YYYY separately)
6. Only map fields where you have matching patient data
7. If a field label mentions multiple options, choose the most appropriate one

OUTPUT FORMAT:
{{
  "field_mappings": {{
    "T12": "John",
    "T13": "Doe", 
    "CB1": true,
    "T14": "01/15/1985"
  }}
}}

Create the mapping now:"""

# Send to Gemini for intelligent mapping
print("🧠 Asking AI to create intelligent field mappings...")
model = genai.GenerativeModel('gemini-2.5-flash')
mapping_response = model.generate_content(auto_mapping_prompt)

print("✅ AI mapping completed!")
print("Raw response preview:")
print(mapping_response.text[:500] + "...")

# Extract and apply the mappings
try:
    if "```json" in mapping_response.text:
        json_start = mapping_response.text.find("```json") + 7
        json_end = mapping_response.text.find("```", json_start)
        mapping_json = mapping_response.text[json_start:json_end].strip()
    else:
        mapping_json = mapping_response.text
    
    ai_mappings = json.loads(mapping_json)
    field_mappings = ai_mappings.get("field_mappings", {})
    
    print(f"🎯 AI created {len(field_mappings)} intelligent field mappings")
    
    # Show sample mappings
    print("Sample mappings:")
    for i, (field, value) in enumerate(list(field_mappings.items())[:5]):
        print(f"  • {field} → {value}")
    
    # Fill PDF with AI mappings
    def fill_pdf_with_ai_mappings(input_pdf_path, output_pdf_path, mappings):
        doc = fitz.open(input_pdf_path)
        filled_count = 0
        
        for page_num in range(len(doc)):
            page = doc[page_num]
            
            for widget in page.widgets():
                field_name = widget.field_name
                
                if field_name in mappings:
                    value = mappings[field_name]
                    
                    try:
                        if widget.field_type == fitz.PDF_WIDGET_TYPE_CHECKBOX:
                            widget.field_value = bool(value)
                        elif widget.field_type == fitz.PDF_WIDGET_TYPE_TEXT:
                            widget.field_value = str(value)
                        
                        widget.update()
                        filled_count += 1
                        print(f"✅ {field_name}: {value}")
                        
                    except Exception as e:
                        print(f"⚠️  Error filling {field_name}: {e}")
        
        doc.save(output_pdf_path)
        doc.close()
        return filled_count
    
    # Apply AI mappings to PDF
    output_pdf_ai = "Input Data/Adbulla/PA_FILLED_AI_MAPPED.pdf"
    filled_count = fill_pdf_with_ai_mappings("Input Data/Adbulla/PA.pdf", output_pdf_ai, field_mappings)
    
    print(f"\n🎉 INTELLIGENT MAPPING SUCCESS!")
    print(f"✅ Filled {filled_count} fields using AI mapping")
    print(f"📄 AI-filled PDF saved as: {output_pdf_ai}")
    
except Exception as e:
    print(f"❌ Error in AI mapping: {e}")
    print("Full AI response:")
    print(mapping_response.text)


🤖 Creating intelligent automated field mapping...
📋 Found 335 fields with labels in PDF
🧠 Asking AI to create intelligent field mappings...
✅ AI mapping completed!
Raw response preview:
```json
{
  "field_mappings": {
    "CB1": false,
    "CB5": true,
    "T12": "Shakh",
    "T13": "Abdulla",
    "T14": "04/01/2001",
    "T15": "425 Sherman Ave, APT D",
    "T16": "Nashville",
    "T17": "TN",
    "T18": "37995",
    "T19": "865-395-3958",
    "T21": "865-395-0481",
    "T28": "LAJM14345116",
    "T29": "435000",
    "T30": "ABDULLA, SHAKH",
    "CB31": false,
    "CB32": true
  }
}
```...
🎯 AI created 16 intelligent field mappings
Sample mappings:
  • CB1 → False
  • CB5 → True
  • T12 → Shakh
  • T13 → Abdulla
  • T14 → 04/01/2001
✅ CB1: False
✅ CB5: True
✅ T12: Shakh
✅ T13: Abdulla
✅ T14: 04/01/2001
✅ T15: 425 Sherman Ave, APT D
✅ T16: Nashville
✅ T17: TN
✅ T18: 37995
✅ T19: 865-395-3958
✅ T21: 865-395-0481
✅ T28: LAJM14345116
✅ T29: 435000
✅ T30: ABDULLA, SHAKH
✅ CB31: False
✅ CB3

In [12]:
# 🚀 IMPROVED INTELLIGENT AUTO-MAPPING with Enhanced Coverage
print("🚀 Creating ENHANCED intelligent automated field mapping...")

# Enhanced mapping prompt that addresses all the gaps
enhanced_mapping_prompt = f"""You are an expert medical form automation specialist. Create comprehensive JSON mapping for a Prior Authorization form.

Pay attention to the whole context of the form and the patient data. And pay attention to this whole prompt including the middle parts

CRITICAL REQUIREMENTS:
1. Fill ALL possible fields where patient data is available
2. Handle checkbox dependencies (if main checkbox is checked, fill related sub-fields)
3. Include product/treatment information from clinical data
4. Use medical terminology appropriately
5. Format dates correctly for each field type
6. Handle multiple choice selections intelligently
7. NEVER use placeholder text like "(From Clinical Data)" - use actual data or leave blank

PDF FIELDS AVAILABLE:
{json.dumps(pdf_fields[:80], indent=2)}

PATIENT DATA AVAILABLE:
{referral_data[:3000]}

SPECIFIC MAPPING INSTRUCTIONS:

A. PATIENT INFORMATION SECTION:
- Map full name to separate first/last name fields
- Map complete address (street, city, state, zip)
- Map all phone numbers (home, work, cell)
- Map DOB in correct format
- Map patient demographics

B. INSURANCE INFORMATION:
- Map member ID, group number, insured name
- Check for secondary coverage and map if available
- Map insurance carrier information

C. PRESCRIBER INFORMATION - CRITICAL RULES:
- Look for "Provider Contact Information" or similar headings in the clinical data
- Search for doctor/physician names in message records or clinical notes for inference if no difinitive contact information is found
- Extract actual prescriber contact details (address, phone, fax)
- Look for NPI numbers, DEA numbers in provider sections
- Check credentials (M.D., D.O., N.P., P.A.) from provider information
- If prescriber information is not clearly available, LEAVE FIELDS BLANK
- DO NOT use placeholder text like "Prescriber Address (From Clinical Data)"

D. TREATMENT/PRODUCT INFORMATION - ENHANCED SEARCH:
- Look for headings containing "Product", "Medication", "Drug", "Treatment"
- Search doctor comments and clinical notes for prescribed medications
- Look for drug names, dosages, frequencies in clinical documentation
- Check for diagnosis codes (ICD-10) in clinical data
- Search for treatment justification in physician notes
- If specific product/medication is not found, LEAVE FIELDS BLANK

E. DISPENSING/ADMINISTRATION:
- Determine administration method from clinical data
- Map pharmacy/dispensing location if specified
- Map administration codes if available

F. CHECKBOX LOGIC:
- If "Start of treatment" is checked, fill start date fields
- If "Continuation of therapy" is checked, fill last treatment date
- For dispensing location, check appropriate option based on data
- For prescriber type, check M.D., D.O., N.P., or P.A. based on credentials

G. DATE FORMATTING:
- For separate MM/DD/YYYY fields, split dates accordingly
- For single date fields, use MM/DD/YYYY format
- Extract dates from clinical timeline

H. MISSING DATA HANDLING - STRICT RULES:
- Only map fields where you have ACTUAL, SPECIFIC subject data
- If information is not clearly available in the clinical data, LEAVE THE FIELD BLANK
- Do not guess, weakly infer, or fabricate information
- Do not use placeholder text or generic descriptions
- Better to have fewer accurate fields than many inaccurate ones

SEARCH STRATEGY FOR PRESCRIBER INFO:
1. Search for "Provider Contact", "Physician Contact", "Doctor Information"
2. Look in message records for sender/physician information
3. Check clinical notes for prescribing physician details
4. If no specific prescriber info found, leave prescriber fields blank

SEARCH STRATEGY FOR PRODUCT INFO:
1. Look for explicit "Product" or "Medication" sections
2. Search doctor comments for drug names and prescriptions
3. Check treatment plans for specific medications
4. Look for pharmaceutical names in clinical documentation
5. If no specific product found, leave product fields blank

OUTPUT FORMAT:
{{
  "field_mappings": {{
    "T12": "Shakh",
    "T13": "Abdulla", 
    "CB1": true,
    "T2": "05",
    "T3": "22", 
    "T4": "2024",
    "CB5": false,
    "T14": "04/01/2001",
    "T15": "425 Sherman Ave APT D",
    "T16": "Nashville",
    "T17": "TN",
    "T18": "37995"
  }},
  "mapping_notes": {{
    "total_fields_mapped": 0,
    "sections_completed": [],
    "missing_data_areas": [],
    "checkbox_decisions": {{}}
  }}
}}

ANALYZE THE CLINICAL DATA THOROUGHLY and create comprehensive mappings using ONLY actual data found."""

# Send enhanced prompt to Gemini
print("🧠 Asking AI for ENHANCED comprehensive field mappings...")
model = genai.GenerativeModel('gemini-2.0-flash')
enhanced_response = model.generate_content(enhanced_mapping_prompt)

print("✅ Enhanced AI mapping completed!")
print("Response preview:")
print(enhanced_response.text[:800] + "...")

# Extract and apply enhanced mappings
try:
    if "```json" in enhanced_response.text:
        json_start = enhanced_response.text.find("```json") + 7
        json_end = enhanced_response.text.find("```", json_start)
        enhanced_json = enhanced_response.text[json_start:json_end].strip()
    else:
        enhanced_json = enhanced_response.text
    
    enhanced_mappings = json.loads(enhanced_json)
    enhanced_field_mappings = enhanced_mappings.get("field_mappings", {})
    mapping_notes = enhanced_mappings.get("mapping_notes", {})
    
    print(f"\n🎯 ENHANCED AI created {len(enhanced_field_mappings)} comprehensive field mappings")
    
    # Show mapping summary
    if mapping_notes:
        print(f"📊 Total fields mapped: {mapping_notes.get('total_fields_mapped', len(enhanced_field_mappings))}")
        print(f"📋 Sections completed: {', '.join(mapping_notes.get('sections_completed', []))}")
        if mapping_notes.get('missing_data_areas'):
            print(f"⚠️  Missing data areas: {', '.join(mapping_notes.get('missing_data_areas', []))}")
    
    # Show sample enhanced mappings
    print("\nSample enhanced mappings:")
    for i, (field, value) in enumerate(list(enhanced_field_mappings.items())[:8]):
        print(f"  • {field} → {value}")
    
    # Apply enhanced mappings to PDF
    output_pdf_enhanced = "Input Data/Adbulla/PA_FILLED_ENHANCED.pdf"
    filled_count = fill_pdf_with_ai_mappings("Input Data/Adbulla/PA.pdf", output_pdf_enhanced, enhanced_field_mappings)
    
    print(f"\n🎉 ENHANCED MAPPING SUCCESS!")
    print(f"✅ Filled {filled_count} fields using enhanced AI mapping")
    print(f"📄 Enhanced PDF saved as: {output_pdf_enhanced}")
    
    # Compare with previous version
    previous_count = len(field_mappings) if 'field_mappings' in locals() else 0
    improvement = filled_count - previous_count
    if improvement > 0:
        print(f"🚀 IMPROVEMENT: +{improvement} more fields filled than previous version!")
    
except Exception as e:
    print(f"❌ Error in enhanced mapping: {e}")
    print("Full enhanced response:")
    print(enhanced_response.text)


🚀 Creating ENHANCED intelligent automated field mapping...
🧠 Asking AI for ENHANCED comprehensive field mappings...
✅ Enhanced AI mapping completed!
Response preview:
```json
{
  "field_mappings": {
    "CB1": true,
    "T2": "05",
    "T3": "22",
    "T4": "2024",
    "T12": "Shakh",
    "T13": "Abdulla",
    "T14": "04/01/2001",
    "T15": "425 Sherman Ave, APT D",
    "T16": "Nashville",
    "T17": "TN",
    "T18": "37995",
    "T19": "865-395-3958",
    "T21": "865-395-0481",
    "T23": "No Known Allergies",
    "T28": "LAJM14345116",
    "T29": "435000"
  },
  "mapping_notes": {
    "total_fields_mapped": 13,
    "sections_completed": [
      "Patient Information Section",
      "Insurance Information"
    ],
    "missing_data_areas": [
      "Prescriber Information",
      "Treatment/Product Information",
	  "Last treatment date"
    ],
    "checkbox_decisions": {
      "CB1": "Start of treatment checked based on common PA rationale",
      "CB5":...

🎯 ENHANCED AI created 16 com